In [0]:
# Cell 1 — Read from Bronze table
bronze_df = spark.table("weather_bronze_raw")

print(f"Total records in Bronze: {bronze_df.count()}")
bronze_df.show(5, truncate=80)

In [0]:
# Cell 2 — Define schema for JSON parsing
from pyspark.sql.types import *

weather_schema = StructType([
    StructField("name", StringType()),
    StructField("dt",   LongType()),
    StructField("main", StructType([
        StructField("temp",       DoubleType()),
        StructField("humidity",   IntegerType()),
        StructField("pressure",   DoubleType()),
        StructField("feels_like", DoubleType()),
        StructField("temp_min",   DoubleType()),
        StructField("temp_max",   DoubleType()),
    ])),
    StructField("wind", StructType([
        StructField("speed", DoubleType()),
        StructField("deg",   IntegerType()),
    ])),
    StructField("weather", ArrayType(StructType([
        StructField("main",        StringType()),
        StructField("description", StringType()),
    ]))),
    StructField("clouds", StructType([
        StructField("all", IntegerType()),
    ])),
    StructField("visibility", IntegerType()),
    StructField("sys", StructType([
        StructField("country", StringType()),
        StructField("sunrise", LongType()),
        StructField("sunset",  LongType()),
    ])),
    StructField("coord", StructType([
        StructField("lat", DoubleType()),
        StructField("lon", DoubleType()),
    ])),
])

print("✓ Schema defined")

In [0]:
# Cell 3 — Parse raw JSON into typed flat columns
from pyspark.sql.functions import (
    from_json, col, from_unixtime, to_timestamp,
    round as spark_round, when
)

parsed_df = bronze_df.withColumn("data", from_json(col("raw_json"), weather_schema))

silver_df = parsed_df.select(
    # Identity
    col("data.name").alias("city"),
    col("data.coord.lat").alias("latitude"),
    col("data.coord.lon").alias("longitude"),
    col("data.sys.country").alias("country"),

    # Temperature
    col("data.main.temp").alias("temp_celsius"),
    col("data.main.feels_like").alias("feels_like_celsius"),
    col("data.main.temp_min").alias("temp_min_celsius"),
    col("data.main.temp_max").alias("temp_max_celsius"),

    # Atmosphere
    col("data.main.humidity").alias("humidity_pct"),
    col("data.main.pressure").alias("pressure_hpa"),
    col("data.visibility").alias("visibility_m"),
    col("data.clouds.all").alias("cloud_cover_pct"),

    # Wind
    col("data.wind.speed").alias("wind_speed_mps"),
    col("data.wind.deg").alias("wind_direction_deg"),

    # Condition
    col("data.weather")[0]["main"].alias("weather_main"),
    col("data.weather")[0]["description"].alias("weather_description"),

    # Timestamps
    to_timestamp(from_unixtime(col("data.dt"))).alias("recorded_at"),
    to_timestamp(from_unixtime(col("data.sys.sunrise"))).alias("sunrise_at"),
    to_timestamp(from_unixtime(col("data.sys.sunset"))).alias("sunset_at"),
    to_timestamp(col("ingested_at")).alias("ingested_at"),
)

print(f"✓ Parsed {silver_df.count()} records")
silver_df.printSchema()

In [0]:
# Cell 4 — Apply data quality filters
total = silver_df.count()

clean_df = silver_df \
    .filter(col("city").isNotNull()) \
    .filter(col("temp_celsius").between(-60, 60)) \
    .filter(col("humidity_pct").between(0, 100)) \
    .filter(col("pressure_hpa").between(870, 1085)) \
    .filter(col("wind_speed_mps") >= 0) \
    .filter(col("recorded_at").isNotNull())

dropped = total - clean_df.count()
print(f"✓ Total records  : {total}")
print(f"✓ Clean records  : {clean_df.count()}")
print(f"✗ Dropped records: {dropped}")

In [0]:
# Cell 5 — Enrich with derived columns
enriched_df = clean_df.withColumn(
    "wind_direction_label",
    when(col("wind_direction_deg") < 22.5,  "N")
    .when(col("wind_direction_deg") < 67.5,  "NE")
    .when(col("wind_direction_deg") < 112.5, "E")
    .when(col("wind_direction_deg") < 157.5, "SE")
    .when(col("wind_direction_deg") < 202.5, "S")
    .when(col("wind_direction_deg") < 247.5, "SW")
    .when(col("wind_direction_deg") < 292.5, "W")
    .when(col("wind_direction_deg") < 337.5, "NW")
    .otherwise("N")
)

print("✓ Derived columns added")
enriched_df.select(
    "city", "temp_celsius", "wind_speed_mps",
    "wind_direction_deg", "wind_direction_label"
).show()

In [0]:
# Cell 6 — Deduplicate incoming batch before writing
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Within each city + recorded_at group, keep only the latest ingested row
# This handles cases where Bronze was accidentally run multiple times
dedup_window = Window \
    .partitionBy("city", "recorded_at") \
    .orderBy(col("ingested_at").desc())

deduped_df = enriched_df \
    .withColumn("rn", row_number().over(dedup_window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

print(f"✓ Before dedup : {enriched_df.count()} rows")
print(f"✓ After  dedup : {deduped_df.count()} rows")
print(f"✗ Duplicates removed: {enriched_df.count() - deduped_df.count()}")

In [0]:
# Cell 7 — Write to Silver Delta table using MERGE (upsert)
# MERGE key: city + recorded_at — unique reading per city per timestamp
# If same city + timestamp already exists → update it (handles API data corrections)
# If new → insert it
from delta.tables import DeltaTable

SILVER_TABLE = "weather_silver_clean"

if not spark.catalog.tableExists(SILVER_TABLE):
    # First run — create the table fresh
    deduped_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(SILVER_TABLE)
    print(f"✓ Silver table created — {deduped_df.count()} rows written")

else:
    # Subsequent runs — MERGE to avoid duplicates
    silver_table = DeltaTable.forName(spark, SILVER_TABLE)

    silver_table.alias("existing").merge(
        deduped_df.alias("new"),
        """
        existing.city        = new.city
        AND existing.recorded_at = new.recorded_at
        """
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

    final_count = spark.table(SILVER_TABLE).count()
    print(f"✓ Silver MERGE complete — {final_count} total rows in table")

In [0]:
# Cell 8 — Verify Silver table
from pyspark.sql.functions import count, countDistinct, min as spark_min, max as spark_max

# Row count
total = spark.table(SILVER_TABLE).count()
print(f"Total rows in Silver: {total}")

# Show latest reading per city
spark.sql(f"""
    SELECT city,
           temp_celsius,
           humidity_pct,
           weather_description,
           wind_speed_mps,
           wind_direction_label,
           recorded_at
    FROM {SILVER_TABLE}
    QUALIFY ROW_NUMBER() OVER (PARTITION BY city ORDER BY recorded_at DESC) = 1
    ORDER BY city
""").show(truncate=False)

# Show reading count per city — confirms history is accumulating correctly
print("\nReadings per city (should grow each pipeline run):")
spark.sql(f"""
    SELECT city,
           COUNT(*)              AS total_readings,
           MIN(recorded_at)      AS first_reading,
           MAX(recorded_at)      AS latest_reading
    FROM {SILVER_TABLE}
    GROUP BY city
    ORDER BY city
""").show(truncate=False)